In [1]:
from googleapi import *
import pprint
import pandas as pd
import os
from config import DATA_DIR
pd.set_option("display.max.columns", None)
import plotly.io as pio
pio.renderers.default = "browser"

In [2]:
cols_to_keep = [
    "name", 
    "id", 
    "types", 
    "formattedAddress", 
    "location", 
    "rating", 
    "googleMapsUri",
    "websiteUri",
    "businessStatus",
    "priceLevel",
    "displayName",
    "takeout",
    "delivery",
    "dineIn",
    "servesBreakfast",
    "servesLunch",
    "servesDinner",
    "servesBrunch",
    "servesDessert",
    "primaryType",
    "reviews",
    "priceRange",
    "postalAddress",
    "editorialSummary",
    "servesVegetarianFood",
]

cph_df = pd.read_json(DATA_DIR / "copenhagen-bounds-limit10_20260316_125502.json")

In [3]:
cph_df.describe()
cph_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4392 entries, 0 to 4391
Data columns (total 66 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   name                          4392 non-null   str    
 1   id                            4392 non-null   str    
 2   types                         4392 non-null   object 
 3   nationalPhoneNumber           3758 non-null   str    
 4   internationalPhoneNumber      3758 non-null   str    
 5   formattedAddress              4392 non-null   str    
 6   addressComponents             4392 non-null   object 
 7   plusCode                      4381 non-null   object 
 8   location                      4392 non-null   object 
 9   viewport                      4392 non-null   object 
 10  rating                        4046 non-null   float64
 11  googleMapsUri                 4392 non-null   str    
 12  websiteUri                    3600 non-null   str    
 13  regularOpening

In [4]:
cph_df.businessStatus.unique()

<ArrowStringArray>
['OPERATIONAL', 'CLOSED_TEMPORARILY']
Length: 2, dtype: str

In [5]:
# only keep operational business statuses 
# removes 192 restaurants that are closed
cph_df = cph_df[cph_df["businessStatus"] == "OPERATIONAL"]
cph_df.info()

<class 'pandas.DataFrame'>
Index: 4200 entries, 0 to 4391
Data columns (total 66 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   name                          4200 non-null   str    
 1   id                            4200 non-null   str    
 2   types                         4200 non-null   object 
 3   nationalPhoneNumber           3613 non-null   str    
 4   internationalPhoneNumber      3613 non-null   str    
 5   formattedAddress              4200 non-null   str    
 6   addressComponents             4200 non-null   object 
 7   plusCode                      4189 non-null   object 
 8   location                      4200 non-null   object 
 9   viewport                      4200 non-null   object 
 10  rating                        3863 non-null   float64
 11  googleMapsUri                 4200 non-null   str    
 12  websiteUri                    3463 non-null   str    
 13  regularOpeningHours

In [6]:
# unwrap the dictionaries in the DF
cph_df["lat"] = cph_df["location"].apply(lambda x: x["latitude"])
cph_df["lon"] = cph_df["location"].apply(lambda x: x["longitude"])
cph_df["displayName"] = cph_df["displayName"].apply(lambda x: x["text"] if isinstance(x, dict) else x)
cph_df["primaryTypeDisplayName"] = cph_df["primaryTypeDisplayName"].apply(lambda x: x["text"] if isinstance(x, dict) else x)

In [7]:
import plotly.express as px
import geopandas as gpd
from geodatasets import get_url

In [8]:
# - plot lat/lon of the data points, and some information
#   - name
#   - main type
#   - price range

def format_price(x):
    if not isinstance(x, dict):
        return "Unknown price range"
    try:
        low = x["startPrice"]["units"]
        high = x["endPrice"]["units"]
        currency = x["startPrice"]["currencyCode"]
        return f"{low} - {high} {currency}"
    except (KeyError, TypeError):
        return "N/A"

cph_df["priceLabel"] = cph_df["priceRange"].apply(format_price)

# cph_df['text'] = cph_df['primaryDisplayName'] + '<br>' + cph_df['primaryTypeDisplayName'] + '<br>' + cph_df['priceLabel']

# Plot
fig = px.scatter_map(
    cph_df,
    lat="lat",
    lon="lon",
    hover_name="displayName",
    hover_data={
        "primaryTypeDisplayName": True,
        "priceLabel": True,
        "lat": False,
        "lon": False,
    },
    color="primaryTypeDisplayName",
    zoom=12,
    center={"lat": 55.6761, "lon": 12.5683},
    title="Copenhagen Restaurants – Google Maps Places",
)

fig.update_layout(map_style="light")
fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig.show()

In [9]:
type_counts_top20 = (
    cph_df["primaryTypeDisplayName"]
    .fillna("Unknown")
    .value_counts()
    .rename_axis("primaryTypeDisplayName")
    .reset_index(name="count")
)

print(type_counts_top20.to_string(index=False))
print(f"\nTotal unique entries: {type_counts_top20.shape[0]}")

     primaryTypeDisplayName  count
                 Restaurant   1459
           Pizza Restaurant    372
         Takeout Restaurant    249
         Italian Restaurant    138
       Fast Food Restaurant    125
           Sushi Restaurant    124
       Hamburger Restaurant    118
            Thai Restaurant     91
                 Kebab Shop     83
                       Cafe     79
              Sandwich Shop     67
          Indian Restaurant     65
                        Bar     60
                     Bakery     54
           Asian Restaurant     53
          Danish Restaurant     45
                Gas Station     43
                      Hotel     43
      Vietnamese Restaurant     42
                   Car Wash     41
                 Juice Shop     41
              Hot Dog Stand     41
         Turkish Restaurant     36
         Chinese Restaurant     35
        Japanese Restaurant     32
         Mexican Restaurant     30
                Coffee Shop     27
                    

In [10]:
cph_df.priceLabel.unique()



<ArrowStringArray>
[      '200 - 300 DKK',       '100 - 200 DKK',         '1 - 100 DKK',
 'Unknown price range',       '100 - 700 DKK',         '1 - 300 DKK',
       '200 - 800 DKK',       '400 - 500 DKK',         '1 - 200 DKK',
       '100 - 300 DKK',       '100 - 600 DKK',       '600 - 700 DKK',
       '300 - 400 DKK',       '100 - 400 DKK',         '1 - 400 DKK',
       '100 - 500 DKK',         '1 - 600 DKK',                 'N/A',
       '300 - 800 DKK',       '200 - 700 DKK',       '200 - 500 DKK',
       '200 - 400 DKK',      '400 - 1000 DKK',       '400 - 900 DKK',
       '300 - 700 DKK',       '300 - 900 DKK',         '1 - 500 DKK',
       '200 - 600 DKK',       '700 - 800 DKK']
Length: 29, dtype: str

In [11]:
import plotly.graph_objects as go

# Unknown price => gray, otherwise color by primary type
is_unknown_price = cph_df["priceLabel"].fillna("").str.strip().str.lower().eq("unknown price range")
type_series = cph_df["primaryTypeDisplayName"].fillna("Unknown type")

# Temporarily mask unknown price rows so they don't get a type color
plot_df = cph_df.copy()
plot_df.loc[is_unknown_price, "primaryTypeDisplayName"] = None

fig = px.scatter_map(
    plot_df,
    lat="lat",
    lon="lon",
    color="primaryTypeDisplayName",
    hover_name="displayName",
    hover_data={"primaryTypeDisplayName": True, "priceLabel": True, "lat": False, "lon": False},
    color_discrete_sequence=px.colors.qualitative.Plotly,
    zoom=9.8,
    center={"lat": 55.68, "lon": 12.53},
    title="Copenhagen Restaurants",
)

# Add gray trace for unknown price entries
unknown_df = cph_df[is_unknown_price]
fig.add_trace(go.Scattermap(
    lat=unknown_df["lat"],
    lon=unknown_df["lon"],
    mode="markers",
    marker=dict(size=8, color="lightgray", opacity=0.8),
    name="Unknown price range",
    text=unknown_df["displayName"],
    customdata=list(zip(unknown_df["primaryTypeDisplayName"], unknown_df["priceLabel"])),
    hovertemplate="<b>%{text}</b><br>Type: %{customdata[0]}<br>Price: %{customdata[1]}<extra></extra>",
))

fig.update_layout(width=800, height=800, margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig.show()

In [12]:
import plotly.express as px
import plotly.graph_objects as go


# --- Prep: cap legend to top 10 types, rest -> "Other" ---
top_types = (
    cph_df["primaryTypeDisplayName"]
    .fillna("Unknown")
    .value_counts()
    .head(25)
    .index.tolist()
)

plot_df = cph_df.copy()
plot_df["typeLabel"] = plot_df["primaryTypeDisplayName"].apply(
    lambda x: x if x in top_types else "Other"
)


# --- Color map: auto-assign Plotly palette to top types, fix special cases ---
color_map = {t: px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)]
             for i, t in enumerate(top_types)}
color_map["Other"] = "#aaaaaa"

# --- Rating label (handle missing) ---
plot_df["ratingLabel"] = plot_df.apply(
    lambda r: f"{r['rating']} ⭐ ({int(r['userRatingCount'])} reviews)"
    if pd.notna(r.get("rating")) and pd.notna(r.get("userRatingCount"))
    else "No rating",
    axis=1,
)

# --- Plot ---
fig = px.scatter_map(
    plot_df,
    lat="lat",
    lon="lon",
    color="typeLabel",
    color_discrete_map=color_map,
    hover_name="displayName",
    hover_data={
        "typeLabel": True,
        "priceLabel": True,
        "ratingLabel": True,
        "lat": False,
        "lon": False,
    },
    zoom=11,
    center={"lat": 55.68, "lon": 12.57},
    title="Copenhagen Restaurants",
    labels={"typeLabel": "Type", "priceLabel": "Price", "ratingLabel": "Rating"},
)

# --- Enable clustering ---
# fig.update_traces(
#     cluster=dict(
#         enabled=True,
#         step=100,        # how many pixels apart points must be to cluster — lower = more aggressive clustering
#         maxzoom=14,     # zoom level at which clustering turns off and individual points appear
#         color="#2563eb", # color of the cluster bubble
#         opacity=0.8,
#     )
# )

fig.update_layout(
    width=800,
    height=800,
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    legend=dict(
        title="Restaurant type",
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="#cccccc",
        borderwidth=1,
    ),
)

fig.show()

In [13]:
from sklearn.cluster import DBSCAN
import numpy as np

# Cluster using DBSCAN on lat/lon
coords = plot_df[["lat", "lon"]].values
kms_per_radian = 6371.0
epsilon = 0.3 / kms_per_radian  # ~300m radius

db = DBSCAN(eps=epsilon, min_samples=5, algorithm="ball_tree", metric="haversine").fit(
    np.radians(coords)
)
plot_df["cluster"] = db.labels_  # -1 = noise (standalone points)

# Split into clustered and standalone
clustered = plot_df[plot_df["cluster"] != -1]
standalone = plot_df[plot_df["cluster"] == -1]

# Build cluster summary
cluster_summary = (
    clustered.groupby("cluster")
    .agg(lat=("lat", "mean"), lon=("lon", "mean"), count=("displayName", "count"))
    .reset_index()
)

# Scale bubble size
cluster_summary["bubbleSize"] = (cluster_summary["count"] / cluster_summary["count"].max() * 50 + 10)

fig = go.Figure()

# Individual points (noise / standalone)
fig.add_trace(go.Scattermap(
    lat=standalone["lat"],
    lon=standalone["lon"],
    mode="markers",
    marker=dict(size=8, color=standalone["typeLabel"].map(color_map), opacity=0.85),
    text=standalone["displayName"],
    customdata=list(zip(standalone["typeLabel"], standalone["priceLabel"], standalone["ratingLabel"])),
    hovertemplate="<b>%{text}</b><br>Type: %{customdata[0]}<br>Price: %{customdata[1]}<br>Rating: %{customdata[2]}<extra></extra>",
    name="Restaurants",
))

# Cluster bubbles
fig.add_trace(go.Scattermap(
    lat=cluster_summary["lat"],
    lon=cluster_summary["lon"],
    mode="markers+text",
    marker=dict(
        size=cluster_summary["bubbleSize"],
        color="#2563eb",
        opacity=0.7,
    ),
    text=cluster_summary["count"].astype(str),
    textfont=dict(color="white", size=11),
    hovertemplate="<b>%{text} restaurants</b><extra></extra>",
    name="Clusters",
))

fig.update_layout(
    width=800,
    height=800,
    map=dict(center=dict(lat=55.68, lon=12.57), zoom=11),
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    title="Copenhagen Restaurants",
)

fig.show()

In [18]:
# Calculate fraction % for each primaryTypeDisplayName
type_counts_top20 = cph_df["primaryTypeDisplayName"].value_counts().head(20)
type_frac = (type_counts_top20 / len(cph_df) * 100).reset_index().round(2)
type_frac = type_frac.rename(columns={"count" : "fraction"})
type_frac

,primaryTypeDisplayName,fraction
0,Restaurant,34.74
1,Pizza Restaurant,8.86
2,Takeout Restaurant,5.93
3,Italian Restaurant,3.29
4,Fast Food Restaurant,2.98
5,Sushi Restaurant,2.95
6,Hamburger Restaurant,2.81
7,Thai Restaurant,2.17
8,Kebab Shop,1.98
9,Cafe,1.88


In [15]:
# Calculate fraction % for each primaryTypeDisplayName
type_counts_top20 = cph_df["primaryTypeDisplayName"].value_counts().head(20)
type_frac_top20 = (type_counts_top20 / len(cph_df) * 100).reset_index().round(2)
type_frac_top20.rename(columns={"count" : "fraction"})

fig = px.bar(
    type_frac_top20,
    x="primaryTypeDisplayName",
    y="fraction",
    title="Top 20 Restaurant Types in Copenhagen",
    labels={
        "primaryTypeDisplayName": "Restaurant Type",
        "fraction": "Fraction (%)"
    },
)

fig.update_traces(
    texttemplate="%{y:.1f}%",
    textposition="outside",
    hovertemplate=None,
    hoverinfo="skip",
)

fig.update_layout(
    xaxis_tickangle=-45,
    yaxis=dict(range=[0, type_frac_top20["fraction"].max() * 1.15]),
    height=600,
    plot_bgcolor="white",
    showlegend=False,
)

fig.show()

ValueError: Value of 'y' is not the name of a column in 'data_frame'. Expected one of ['primaryTypeDisplayName', 'count'] but received: fraction